# All Problems

## <u>**Problem 0**</u> 

Create GitHub Issues for the following problems, either using one issue per problem or breaking problems down into appropriate sub-issues.

Use the issues to track your progress. Comment on your progress as you work and close each issue when you consider the corresponding problem complete.


<br>
<br>
<br>
<br>
<br>












## <u>**Problem 1 - Representing the SHA-256 Data**</u>

#### **1.1 Representing 32-bit Words**

SHA-256 operates on 32-bit words, represented as unsigned integers ranging from (0) to (2^{32}-1). In Python, I will use the built-in int type because it supports arithmetic and bitwise operations. However, Python integers have arbitrary precision, so I will use the mask 0xFFFFFFFF to restrict results to 32 bits when necessary. Eight hexadecimal digits represent one 32-bit word.


The following example demonstrates how to represent a 32-bit word
and enforce the 32-bit limit using a bit mask.

In [1]:
# Define the 32-bit word mask
WORD_MASK = 0xFFFFFFFF

# Represent a 32-bit word
word = 0x12345678

print(f"Word in hexadecimal: {word:08x}")
print(f"Word in decimal: {word}")
print(f"Maximum 32-bit unsigned value: {WORD_MASK}")

# Demonstrate 32-bit arithmetic
result = (WORD_MASK + 1) & WORD_MASK
print(f"Result after 32-bit overflow: {result:08x}")

Word in hexadecimal: 12345678
Word in decimal: 305419896
Maximum 32-bit unsigned value: 4294967295
Result after 32-bit overflow: 00000000


#### Explanation

The value `0x12345678` represents a 32-bit word in hexadecimal notation.
The mask `0xFFFFFFFF` contains 32 one-bits and retains the lowest 32
bits of a value when applied using the bitwise AND operator (`&`).

Python integers can exceed 32 bits, so masking is necessary to reproduce
SHA-256's arithmetic modulo \(2^{32}\). As demonstrated, adding one to
the maximum unsigned 32-bit value produces zero after masking.

#### References

1. NIST (2015). *Secure Hash Standard (FIPS PUB 180-4)*,
   Sections 3.1–3.2.
   https://doi.org/10.6028/NIST.FIPS.180-4

2. Python Software Foundation. *Python Documentation: Bitwise
   Operations on Integer Types*.
   https://docs.python.org/3/library/stdtypes.html#bitwise-operations-on-integer-types

<br>


#### **1.2 Representing sequences of 32-bit words**

SHA-256 uses sequences of 32-bit words when processing message blocks and performing its internal calculations. I will represent these sequences using Python lists, which preserve the order of elements and allow individual words to be accessed by index. Each element will be a Python integer restricted to 32 bits using the mask 0xFFFFFFFF when necessary.

A list is appropriate because SHA-256 requires words to be processed in a specific order, and its message schedule contains 64 words.

In [2]:
# Define the 32-bit word mask
WORD_MASK = 0xFFFFFFFF

# Represent a sequence of 32-bit words using a list
words = [
    0x12345678,
    0x9ABCDEF0,
    0x0F1E2D3C,
    0xAABBCCDD
]

# Display the sequence
print("Number of words:", len(words))
print("First word:", f"{words[0]:08X}")
print("Second word:", f"{words[1]:08X}")

# Demonstrate enforcing the 32-bit limit
words.append((0xFFFFFFFF + 1) & WORD_MASK)

print("Last word after masking:", f"{words[-1]:08X}")

Number of words: 4
First word: 12345678
Second word: 9ABCDEF0
Last word after masking: 00000000


#### Explanation

The list preserves the order of the words and allows individual elements to be accessed using indexing. The `len()` function returns the number of elements, while the mask ensures that the calculated result remains within the 32-bit range.

This representation will be useful when storing the 16 words of a message block and the 64 words of the SHA-256 message schedule.

#### References

1. National Institute of Standards and Technology (NIST). (2015).
   *Secure Hash Standard (FIPS PUB 180-4)*, Section 6.2.2.
   https://doi.org/10.6028/NIST.FIPS.180-4

2. Python Software Foundation. *Python 3 Documentation: Data Structures — More on Lists*.
   https://docs.python.org/3/tutorial/datastructures.html

<br>

#### **1.3 Representing Input Messages**


SHA-256 processes a message as a sequence of bits. In Python, I will represent input messages using the built-in bytes type, which stores an immutable sequence of byte values from 0 to 255. Text must be encoded into bytes before it can be processed; for this example, I will use UTF-8 encoding. This is consistent with the input processing described in the [NIST Secure Hash Standard (FIPS 180-4)](https://doi.org/10.6028/NIST.FIPS.180-4). The [official Python documentation on bytes objects](https://docs.python.org/3/library/stdtypes.html#bytes-objects) explains this representation.



In [1]:
# Define a text message
message = "Hello, SHA-256!"

# Encode the text as UTF-8 bytes
message_bytes = message.encode("utf-8")

# Display the original message and its byte representation
print("Original message:", message)
print("Encoded message:", message_bytes)

# Display the message length in bytes and bits
print("Message length in bytes:", len(message_bytes))
print("Message length in bits:", len(message_bytes) * 8)

Original message: Hello, SHA-256!
Encoded message: b'Hello, SHA-256!'
Message length in bytes: 15
Message length in bits: 120


#### Explanation

The `encode("utf-8")` method converts the text string into a sequence of bytes. The `len()` function returns the number of bytes in that sequence. Since each byte contains eight bits, the 15-byte message has a length of 120 bits.

Using `bytes` provides a suitable representation for SHA-256 input because it supports both encoded text and arbitrary binary data. This allows messages to be processed consistently, regardless of whether they originally represent text or binary content.

#### References

- National Institute of Standards and Technology (NIST). *Secure Hash Standard (SHS), FIPS PUB 180-4*. This standard defines the SHA-256 algorithm and its message-processing requirements. [Available online](https://doi.org/10.6028/NIST.FIPS.180-4).

- Python Software Foundation. *Built-in Types — Bytes Objects*. This documentation explains Python's immutable bytes type and how it represents binary data. [Available online](https://docs.python.org/3/library/stdtypes.html#bytes-objects).

<br>


#### **1.4 Representing 512-bit Blocks**

SHA-256 processes messages in blocks of 512 bits, equivalent to 64 bytes. I will represent each block using Python's built-in `bytes` type. Each block is divided into sixteen 32-bit words, represented as unsigned integers.

The [NIST Secure Hash Standard (FIPS 180-4), Section 6.2.2](https://doi.org/10.6028/NIST.FIPS.180-4) specifies the SHA-256 message-block processing procedure. Python's [`int.from_bytes()` documentation](https://docs.python.org/3/library/stdtypes.html#int.from_bytes) explains how bytes can be converted into integers using a specified byte order.

The following example demonstrates how to represent a 64-byte block and convert it into sixteen 32-bit words using big-endian byte order.

In [1]:
# Create a 64-byte block containing values from 0 to 63
block = bytes(range(64))

# Display the block length in bytes and bits
print("Block length in bytes:", len(block))
print("Block length in bits:", len(block) * 8)

# Divide the block into sixteen 32-bit words
words = []

for i in range(0, len(block), 4):
    # Convert four bytes into one big-endian 32-bit word
    word = int.from_bytes(block[i:i + 4], byteorder="big")
    words.append(word)

# Display the number of words and the first two words
print("Number of 32-bit words:", len(words))
print("First word:", f"0x{words[0]:08X}")
print("Second word:", f"0x{words[1]:08X}")

Block length in bytes: 64
Block length in bits: 512
Number of 32-bit words: 16
First word: 0x00010203
Second word: 0x04050607


#### Explanation

The expression `bytes(range(64))` creates a sequence of 64 bytes, with values ranging from 0 to 63. Since each byte contains eight bits, the block represents 512 bits.

The loop processes four bytes at a time. The `int.from_bytes()` function combines each group into one 32-bit integer using big-endian byte order, meaning the most significant byte comes first. For example, the bytes `00 01 02 03` become the word `0x00010203`.

The resulting list contains sixteen 32-bit words, which is the representation required for a SHA-256 message block.

#### References

- National Institute of Standards and Technology (NIST). *Secure Hash Standard (SHS), FIPS PUB 180-4*, Section 6.2.2. Specifies SHA-256 message-block processing and the use of sixteen 32-bit words. [Available Here](https://doi.org/10.6028/NIST.FIPS.180-4).

- Python Software Foundation. *Built-in Types — int.from_bytes()*. Documents how to convert bytes into integers using a specified byte order. [Available Here](https://docs.python.org/3/library/stdtypes.html#int.from_bytes).

## <u>**Problem 2 - Representing the SHA-256 Data**</u>

## Problem 3: Generating the SHA-256 Constants

## Problem 4: Padding and Parsing Messages


## Problem 5: The SHA-256 Compression Function

## Problem 6: Complete SHA-256